# In this notebook we choose which pre-process steps to take for the clustering task
We will choose between the following steps:
* vectorization technique (BoW or TF-IDF)
* stemmization/lemmization
* number of stopwords (50, 100, 150)
We will test by executing KMEANS 

In [ ]:
# %pip install nltk
# import nltk
# nltk.download('punkt')
# nltk.download('wordnet')
# %pip install langdetect

Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package punkt to
[nltk_data]     /home/students/s346259/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/students/s346259/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np

from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.stem import LancasterStemmer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from langdetect import detect, DetectorFactory

In [3]:
def remove_rare_words(data):
    #remove all of the words that appear only once in the entire dataset
    all_words = ' '.join(data['body']).split()
    word_freq = pd.Series(all_words).value_counts()
    rare_words = word_freq[word_freq == 1].index
    data['body'] = data['body'].apply(lambda x: ' '.join([word for word in x.split() if word not in rare_words]))
    del all_words, word_freq, rare_words
    return data

data = pd.read_csv('data_unsupervised.csv')

#This steps are "fixed" in the sense that they will always be done in this way
data['body'] = data['body'].str.lower()#to lowercase
data['body'] = data['body'].str.replace(r'http\S+|www\.\S+', '', regex=True)#remove URLs
data['body'] = data['body'].str.replace(r'[^a-z\s]', '', regex=True)#remove everything except letters (english alphabet) and whitespace
data = data[data['body'].str.strip().astype(bool)]#remove comments with only whitespace
data = data.dropna()#remove empty comments
data=data.drop_duplicates() #removing duplicates

data = remove_rare_words(data)

data = data[data['body'].str.strip().astype(bool)]#remove comments with only whitespace (again after we removed the words)


In [ ]:
#keep only comments in english

langs = []
try:
    with open("unsupervised_lang.txt") as f:#Just to avoid re-detecting languages every time
        for line in f:
            langs.append(line.strip())
        data = data[pd.Series(langs, index=data.index)=='en']
except Exception as e :
    print(e)
    DetectorFactory.seed = 0
    for text in data['body']:
        # if not text.strip():
        #     langs.append("empty")
        # else:
        try:
            langs.append(detect(text))
        except:
            langs.append("error")
    data = data[pd.Series(langs, index=data.index)=='en']
    with open("unsupervised_lang.txt", "w") as f:
        for lang in langs:
                f.write(f"{lang}\n")

[Errno 2] No such file or directory: 'unsupervised_lang.txt'


In [5]:
def remove_stopwords(numerStopwords, df):
    MostFrequentWords = df['body'].str.split().explode().value_counts()[0:numerStopwords].index.tolist()
    #print(MostFrequentWords)
    #removing most frequent words
    for word in MostFrequentWords:
        df['body']=df['body'].str.replace(rf'\b{word}\b', '', regex=True)
    return df

def lemmatize_text(df):
    lemmatizer = WordNetLemmatizer()
    df['body'] = df['body'].apply(lambda x: ' '.join([lemmatizer.lemmatize(word) for word in x.split()]))
    return df

def stem_text(df):
    stemmer = LancasterStemmer()
    df['body'] = df['body'].apply(lambda x: ' '.join([stemmer.stem(word) for word in x.split()]))
    return df

In [6]:
Tokenizers = {
    'stemmer': LancasterStemmer(),
    'lemmatizer': WordNetLemmatizer()
}

Vectorizers = {
    'BoW': CountVectorizer(),
    'tfidf': TfidfVectorizer()
}

number_of_stopswords=[50,100,150]

for numerStopwords in number_of_stopswords:
    without_stopwords = remove_stopwords(numerStopwords, df=data.copy())
    
    for tokenizer_name, tokenizer in Tokenizers.items():
        if tokenizer_name == 'lemmatizer':
            tokenized = lemmatize_text(without_stopwords.copy())
        else:
            tokenized = stem_text(without_stopwords.copy())
        
        for vectorizer_name, vectorizer in Vectorizers.items():
            print(f'Processing with {numerStopwords} stopwords, {tokenizer_name} and {vectorizer_name}')

            vectorized = vectorizer.fit_transform(tokenized['body'])

            # Dimensionality Reduction
            #n_components = min(2000, vectorized.shape[1]-1, vectorized.shape[0]-1)
            n_components = 1000
            svd = TruncatedSVD(n_components=n_components, random_state=42)
            X_reduced = svd.fit_transform(vectorized)
            variance = svd.explained_variance_ratio_
            if np.cumsum(variance)[-1] < 0.75:
                print(f'Warning: only {np.cumsum(variance)[-1]:.2f} variance explained with {n_components} components')
            else:
                n_components_75 = np.argmax(np.cumsum(variance) >= 0.75) + 1
                svd = TruncatedSVD(n_components=n_components_75, random_state=42)
                X_reduced = svd.fit_transform(vectorized)
                print(f'{np.cumsum(variance)[n_components_75-1]:.2f} variance explained with {n_components_75} components')
            
            #Store X_reduced as a file
            np.save(f"X_reduced_{numerStopwords}_{tokenizer_name}_{vectorizer_name}.npy", X_reduced)

            kmeans = KMeans(
                n_clusters=5,
                init='k-means++', #centroids are equally spaced -> probably better than random initialization. and should converge faster
                random_state=42, 
                n_init='auto', #number of times the k-means algorithm will be run with different centroid seeds. 'auto' uses a default value based on init
                max_iter=100)
            kmeans.fit(X_reduced)
            labels = kmeans.labels_
            silhouette_avg = silhouette_score(X_reduced, labels, sample_size=10000)
            print(f'Silhouette score for {numerStopwords} stopwords, {tokenizer_name} and {vectorizer_name}: {silhouette_avg}')


Processing with 50 stopwords, stemmer and BoW
0.75 variance explained with 828 components
Silhouette score for 50 stopwords, stemmer and BoW: 0.3189556094962652
Processing with 50 stopwords, stemmer and tfidf
Silhouette score for 50 stopwords, stemmer and tfidf: 0.010747118062299976
Processing with 50 stopwords, lemmatizer and BoW
Silhouette score for 50 stopwords, lemmatizer and BoW: 0.3114239572356378
Processing with 50 stopwords, lemmatizer and tfidf
Silhouette score for 50 stopwords, lemmatizer and tfidf: 0.011993538874635456
Processing with 100 stopwords, stemmer and BoW
Silhouette score for 100 stopwords, stemmer and BoW: 0.31181018164175267
Processing with 100 stopwords, stemmer and tfidf
Silhouette score for 100 stopwords, stemmer and tfidf: 0.008558638420769225
Processing with 100 stopwords, lemmatizer and BoW
Silhouette score for 100 stopwords, lemmatizer and BoW: 0.30019801313660693
Processing with 100 stopwords, lemmatizer and tfidf
Silhouette score for 100 stopwords, lemma